## Проект: Retrieval-Augmented Generation (RAG) на данных StackOverflow
## Описание проекта

Цель проекта — построить систему поиска и генерации ответов на основе данных StackOverflow с использованием подхода Retrieval-Augmented Generation (RAG).

В рамках проекта реализован полный пайплайн подготовки данных, разбиения документов на чанки, генерации эмбеддингов и хранения их в векторной базе данных PostgreSQL с расширением pgvector.

Подход RAG позволяет находить релевантные фрагменты текста с помощью векторного поиска и использовать их в качестве контекста для генерации ответа языковой моделью.

In [1]:
import pandas as pd
from src.ingestion.loader import CSVLoader, DataLoader
from src.core.config import Settings
from src.ingestion.chunker import Chunker
from src.indexing.embedder import encode
from src.ingestion.cleaner import Preprocessing
from nltk.tokenize import word_tokenize
from pathlib import Path

#import torch
#from sentence_transformers import SentenceTransformer

#device = 'cuda' if torch.cuda.is_available() else 'cpu'
#model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

loader_csv = CSVLoader()
settings = Settings()
preprocessing = Preprocessing()
chunker = Chunker()
loader_data = DataLoader()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 1. Подготовка данных

Загружаем три файла данных: Question, Answers и Tags. 

In [ ]:
data_question = loader_csv.load_csv(settings.path_question)
data_answers = loader_csv.load_csv(settings.path_answers)
data_tags = loader_csv.load_csv(settings.path_tags)

### 2. Обработка данных

Производим обработку данных:
* Объединение данных - Question, Answers и Tags,
* Выбор только лучшего ответа
* Очистка текста от html-тегов
* Формирование документа
* Очистка от избыточной информации

In [ ]:
data = preprocessing.execute(data_question, data_answers, data_tags)

In [ ]:
data.head(5)

Делаем контрольную точку

In [ ]:
data.to_parquet("data/processed/rag_documents.parquet", index=False)

Освобождаем память от датафрейма

In [ ]:
del data

### 3. Chunking

Загружаем контрольную точку перед чанкером.

In [15]:
df = pd.read_parquet("data/processed/rag_documents.parquet")

In [16]:
df.head(5)

,Id,Score_question,Title,Id_answer,Score_answer,Tag,document_text,doc_length_words
0,80,26,SQLStatement.execute() - multiple queries in o...,124,12,"actionscript-3, air, flex",Title: SQLStatement.execute() - multiple queri...,410
1,90,144,Good branching and merging tutorials for Torto...,1466832,19,"branch, branching-and-merging, svn, tortoisesvn",Title: Good branching and merging tutorials fo...,77
2,120,21,ASP.NET Site Maps,124363,9,"asp.net, sitemap, sql","Title: ASP.NET Site Maps\n\nTags: asp.net, sit...",301
3,180,53,Function for creating color wheels,539,21,"algorithm, color-space, colors, language-agnostic",Title: Function for creating color wheels\n\nT...,283
4,260,49,Adding scripting functionality to .NET applica...,307,28,".net, c#, compiler-construction, scripting",Title: Adding scripting functionality to .NET ...,393


Режем документ на части, и добавляем столбец по количеству чанков.

In [17]:
df["chunks"] = df["document_text"].apply(chunker.chunk_document)
df["chunk_count"] = df["chunks"].apply(len)

In [18]:
df.head(5)

,Id,Score_question,Title,Id_answer,Score_answer,Tag,document_text,doc_length_words,chunks,chunk_count
0,80,26,SQLStatement.execute() - multiple queries in o...,124,12,"actionscript-3, air, flex",Title: SQLStatement.execute() - multiple queri...,410,[Title : SQLStatement.execute ( ) - multiple q...,4
1,90,144,Good branching and merging tutorials for Torto...,1466832,19,"branch, branching-and-merging, svn, tortoisesvn",Title: Good branching and merging tutorials fo...,77,[Title : Good branching and merging tutorials ...,1
2,120,21,ASP.NET Site Maps,124363,9,"asp.net, sitemap, sql","Title: ASP.NET Site Maps\n\nTags: asp.net, sit...",301,"[Title : ASP.NET Site Maps Tags : asp.net , si...",3
3,180,53,Function for creating color wheels,539,21,"algorithm, color-space, colors, language-agnostic",Title: Function for creating color wheels\n\nT...,283,[Title : Function for creating color wheels Ta...,3
4,260,49,Adding scripting functionality to .NET applica...,307,28,".net, c#, compiler-construction, scripting",Title: Adding scripting functionality to .NET ...,393,[Title : Adding scripting functionality to .NE...,4


Отделяем чанки на разные строки и формируем уникальный столбец `chunk_id`.

In [20]:
chunks_df = chunker.build_chunks_df(df)

In [ ]:
del df

In [21]:
chunks_df.head(5)

,chunk_id,question_id,answer_id,chunk_index,title,tags,question_score,answer_score,chunk_text
0,q80_a124_c0,80,124,0,SQLStatement.execute() - multiple queries in o...,"actionscript-3, air, flex",26,12,Title : SQLStatement.execute ( ) - multiple qu...
1,q80_a124_c1,80,124,1,SQLStatement.execute() - multiple queries in o...,"actionscript-3, air, flex",26,12,"Primary Key , categoryName varchar ( 50 ) , pa..."
2,q80_a124_c2,80,124,2,SQLStatement.execute() - multiple queries in o...,"actionscript-3, air, flex",26,12,strSQL ; sqlStatement.sqlConnection = sqlConne...
3,q80_a124_c3,80,124,3,SQLStatement.execute() - multiple queries in o...,"actionscript-3, air, flex",26,12,= stream.readUTFBytes ( stream.bytesAvailable ...
4,q90_a1466832_c0,90,1466832,0,Good branching and merging tutorials for Torto...,"branch, branching-and-merging, svn, tortoisesvn",144,19,Title : Good branching and merging tutorials f...


Делаем контрольную точку.

In [22]:
chunks_df.to_parquet("data/processed/rag_documents_chunker.parquet", index=False)

Освобождаем память от датафрейма

In [ ]:
del chunks_df

### 4. Embeddings

Загружаем предыдущую контрольную точку.

In [2]:
chunks_df = pd.read_parquet("data/processed/rag_documents_chunker.parquet")

Формируем эмбеддинги из чанков 

In [3]:
embeddings = encode(chunks_df['chunk_text'].tolist(), batch_size=256, show_progress_bar=False, convert_to_numpy=True)

In [4]:
chunks_df['embedding'] = list(embeddings)

In [5]:
chunks_df.head(5)

,chunk_id,question_id,answer_id,chunk_index,title,tags,question_score,answer_score,chunk_text,embedding
0,q80_a124_c0,80,124,0,SQLStatement.execute() - multiple queries in o...,"actionscript-3, air, flex",26,12,Title : SQLStatement.execute ( ) - multiple qu...,"[0.029834066, -0.03355406, -0.10831062, 0.1178..."
1,q80_a124_c1,80,124,1,SQLStatement.execute() - multiple queries in o...,"actionscript-3, air, flex",26,12,"Primary Key , categoryName varchar ( 50 ) , pa...","[-0.022446949, -0.030678289, -0.1353806, 0.061..."
2,q80_a124_c2,80,124,2,SQLStatement.execute() - multiple queries in o...,"actionscript-3, air, flex",26,12,strSQL ; sqlStatement.sqlConnection = sqlConne...,"[0.01638568, -0.036768716, -0.047633044, 0.076..."
3,q80_a124_c3,80,124,3,SQLStatement.execute() - multiple queries in o...,"actionscript-3, air, flex",26,12,= stream.readUTFBytes ( stream.bytesAvailable ...,"[0.055042792, 0.033023063, -0.0871628, -0.0128..."
4,q90_a1466832_c0,90,1466832,0,Good branching and merging tutorials for Torto...,"branch, branching-and-merging, svn, tortoisesvn",144,19,Title : Good branching and merging tutorials f...,"[0.007835973, -0.06721794, -0.03726018, -0.053..."


Делаем контрольную точку.

In [9]:
#chunks_df.to_parquet("data/processed/rag_documents_embeddings.parquet", index=False)

In [8]:
chunk_size = 100000  # Подберите размер в зависимости от доступной памяти
for i in range(0, len(chunks_df), chunk_size):
    chunks_df.iloc[i:i+chunk_size].to_parquet(f"data/processed/rag_documents_embeddings_{i//chunk_size}.parquet", index=False)

In [10]:
del chunks_df

### 5. Сохранение данных в БД

In [2]:
for path in sorted(Path("data/processed").glob("rag_documents_embeddings_*.parquet")):
    df = pd.read_parquet(path)
    loader_data.insert_to_db(df)
    print(f"loaded: {path.name}")

loaded: rag_documents_embeddings_0.parquet
loaded: rag_documents_embeddings_1.parquet
loaded: rag_documents_embeddings_10.parquet
loaded: rag_documents_embeddings_11.parquet
loaded: rag_documents_embeddings_12.parquet
loaded: rag_documents_embeddings_13.parquet
loaded: rag_documents_embeddings_14.parquet
loaded: rag_documents_embeddings_15.parquet
loaded: rag_documents_embeddings_16.parquet
loaded: rag_documents_embeddings_17.parquet
loaded: rag_documents_embeddings_18.parquet
loaded: rag_documents_embeddings_19.parquet
loaded: rag_documents_embeddings_2.parquet
loaded: rag_documents_embeddings_20.parquet
loaded: rag_documents_embeddings_21.parquet
loaded: rag_documents_embeddings_22.parquet
loaded: rag_documents_embeddings_23.parquet
loaded: rag_documents_embeddings_24.parquet
loaded: rag_documents_embeddings_25.parquet
loaded: rag_documents_embeddings_26.parquet
loaded: rag_documents_embeddings_27.parquet
loaded: rag_documents_embeddings_28.parquet
loaded: rag_documents_embeddings_29

### 6. Поиск

In [ ]:
query = 'SELECT'
query_embedding = encode(query)

doc = loader_data.get_data_from_db(query_embedding)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Result 1:
Chunk ID: q16412050_a16412083_c1
Similarity: 0.0897
Chunk Text: SELECT *...
--------------------------------------------------
Result 2:
Chunk ID: q33159370_a33160144_c3
Similarity: 0.1308
Chunk Text: option in select...
--------------------------------------------------
Result 3:
Chunk ID: q32834890_a32834958_c2
Similarity: 0.1403
Chunk Text: SELECT ......
--------------------------------------------------
Result 4:
Chunk ID: q8991940_a8992142_c2
Similarity: 0.1734
Chunk Text: a select ?...
--------------------------------------------------
Result 5:
Chunk ID: q13705700_a13705774_c2
Similarity: 0.2032
Chunk Text: 'select * from ... '...
--------------------------------------------------
Result 6:
Chunk ID: q26573070_a26573863_c4
Similarity: 0.2060
Chunk Text: /select >...
--------------------------------------------------
Result 7:
Chunk ID: q38548470_a38556742_c4
Similarity: 0.2060
Chunk Text: /select >...
--------------------------------------------------
Result 8:
Chunk I

In [3]:
doc

[('q31996000_a31996027_c2',
  "select 'abc ' from dual ) ; X -- -- -- ab ab ...",
  0.3403728407732496),
 ('q10840270_a10844283_c2', 'abc', 0.3407268917127304),
 ('q26263450_a26263740_c2', 'abc2 abc3', 0.3674194812774658),
 ('q21296000_a21296768_c3',
  'select > < option value=1 > abc < /option > < option value=567 > hello < /option > < option value=91 > hjk < /option > < /select >',
  0.3674851655960083),
 ('q22627540_a22628009_c4', "( 3 ) `` abc '' }", 0.37448666247804874),
 ('q11551820_a11554592_c1',
  "' , 'ABC ' , 'ABCDEFGHIJKLM ' ; SQL will receive ABC ABCDEFGHIJ ABC ABCD but with no other formatings like borders etc , so very rudimentary ant",
  0.3841844613607065),
 ('q15378610_a15378653_c1',
  '-- -- - | 4 | 7 | 4 | -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- - And other table tbl_looksup which looks like -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- | id | language | value | -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- | 1 | 1 | abc | -- -- -- -- -- -- -- -- -- -- -- --

In [4]:
type(query_embedding)

list